In [1]:
# --- CLEAN INSTALL FOR COLAB + T4 GPU + CUDA 12.1 ---

# 1. Uninstall broken/conflicting libraries
!pip uninstall -y bitsandbytes triton torch torchvision torchaudio

# 2. Install PyTorch compatible with CUDA 12.1 (perfect for Colab T4)
!pip install torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3. Install bitsandbytes version compatible with CUDA 12 + PyTorch 2.3
!pip install bitsandbytes==0.45.0

# 4. Install transformers + accelerate + peft
!pip install transformers accelerate peft

# 5. Print versions (debug)
import torch, bitsandbytes as bnb
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)


Found existing installation: torch 2.3.1+cu121
Uninstalling torch-2.3.1+cu121:
  Successfully uninstalled torch-2.3.1+cu121
Found existing installation: torchvision 0.18.1+cu121
Uninstalling torchvision-0.18.1+cu121:
  Successfully uninstalled torchvision-0.18.1+cu121
Found existing installation: torchaudio 2.3.1+cu121
Uninstalling torchaudio-2.3.1+cu121:
  Successfully uninstalled torchaudio-2.3.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.3.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.9 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached https://download.pyt

In [2]:
from huggingface_hub import login
login()

mapping = {}

mapping["periodontal_abscess"] = {
    "false toothache",
    "gum pain",
    "pain when touched",
    "sensitivity when biting",
    "unusual taste",
    "salty-tasting fluid",
    "gum swelling with pus",
    "pain on palpation",
}

mapping["simple_cavities"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
    "pain to sour",
    "discomfort when brushing",
    "pain to stimuli",
}

mapping["acute_apical_periodontitis"] = {
    "pain when chewing",
    "pain when touched",
    "tooth feels higher",
    "mobile tooth",
    "discomfort on palpation",
    "pain on percussion",
}

mapping["chronic_apical_periodontitis"] = {
    "gum swelling",
    "salty-tasting fluid",
    "pressure when biting",
    "dull ache in the tooth",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

mapping["pericoronitis"] = {
    "continuous pain in the wisdom tooth area",
    "pain when chewing",
    "pain when swallowing",
    "cannot open mouth fully",
    "swelling over the wisdom tooth",
    "inflamed gum",
    "partially erupted wisdom tooth",
    "salty-tasting fluid",
}

mapping["reversible_pulpitis"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
}

mapping["irreversible_pulpitis"] = {
    "spontaneous pain",
    "strong prolonged pain",
    "pain to cold",
    "pain to heat",
    "throbbing pain radiating to the ear",
    "slight pain on percussion",
    "tolerable sensitivity on palpation",
}

mapping["pulp_necrosis"] = {
    "spontaneous pain",
    "intense short pain",
    "pain to heat",
    "pressure when biting",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

lora_dir = r"/content/drive/My Drive/pacient-llama31-8b-lora-FINAL/checkpoint-52"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # pe multe GPU-uri e mai safe decât bfloat16
)

torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map={"": 0} if device == "cuda" else None,  # NU mai folosim "auto"
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False

model = PeftModel.from_pretrained(
    base_model,
    lora_dir,
)

model.eval()
print("Model + LoRA încărcate!")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Model + LoRA încărcate!


In [28]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Set
import random
import torch

SYSTEM_PROMPT_TEMPLATE = """
You are NOT a language model in this exercise.
You are playing the role of a REAL HUMAN PATIENT in a medical simulation scenario.

Your role:
- You are a patient talking to a dental student.
- Your goal is to help the student practice identifying symptoms.
- You do not know diagnoses, analyze causes, explain medicine. You just tell how you feel.

INSIDE INFORMATION (only for you, do not reveal it):
- Real diagnosis (hidden): {diagnosis}
- Complete and final list of your real symptoms:
{symptom_bullets}

ABSOLUTE RULES (take precedence over any other instructions):
1. The symptoms in the list above represent your REALITY.
- If the student asks you about a symptom FROM THE LIST → answer YES, you have it.
- If they ask about something that is NOT on the list → you answer NO, you do not have that symptom.
(This rule is mandatory and cannot be broken.)

2. Never invent new symptoms.
3. Do not add unsolicited symptoms unless the student explicitly asks
“Do you have any other problems?” or something similar.
4. Do not use the labels on the list robotically.
Transform the symptoms into natural, realistic sentences.
5. Answer only the student’s question, in 1–3 short sentences.
6. Never say the diagnosis, medical causes or specialist terms.
7. Your role is TO BE PATIENT. Do not step out of role under any circumstances.
""".strip()



def build_system_prompt_for_case(disease_key: str, mapping: Dict[str, set]) -> str:
    """
    Construiește system prompt-ul final pentru un caz, pe baza:
      - cheii bolii (ex: 'carie_simpla')
      - mapping-ului boala -> set de simptome
    """
    diagnosis = disease_key.replace("_", " ")
    symptoms = sorted(list(mapping.get(disease_key, [])))

    if symptoms:
        symptom_bullets = "\n".join(f"- {s}" for s in symptoms)
    else:
        symptom_bullets = "- (nu sunt definite simptome pentru acest caz)"

    return SYSTEM_PROMPT_TEMPLATE.format(
        diagnosis=diagnosis,
        symptom_bullets=symptom_bullets,
    )


In [5]:
from dataclasses import dataclass, field
import random

@dataclass
class Case:
    diagnosis_truth: str
    symptoms_truth: Set[str]
    revealed_symptoms: Set[str] = field(default_factory=set)


@dataclass
class State:
    history: List[Dict[str, str]] = field(default_factory=list)
    memory_summary: str = ""
    turns_since_summary: int = 0


def init_case(mapping: Dict[str, set]) -> Case:
    disease_key = random.choice(list(mapping.keys()))
    symptoms = set(mapping[disease_key])
    return Case(
        diagnosis_truth=disease_key,
        symptoms_truth=symptoms,
    )


In [56]:
def format_history_for_summary(history, max_messages=20):
    recent = history[-max_messages:]
    lines = []
    for turn in recent:
        if turn["role"] == "assistant":
            lines.append(f"[PATIENT] {turn['content']}")
        elif turn["role"] == "user":
            lines.append(f"[DOCTOR] {turn['content']}")
    return "\n".join(lines)


SUMMARY_PROMPT = """
You are a medical memory updater.

Your goal is to maintain a structured memory of the PATIENT's dental symptoms.

RULES:
- Extract ONLY information directly stated by the PATIENT.
- Ignore DOCTOR messages unless they force the patient to reveal new info.
- ALWAYS merge new info with PREVIOUS_MEMORY.
- NEVER invent anything.
- NEVER delete older info unless the patient contradicts it.
- NEVER output explanations.
- NEVER output JSON.
- Output ONLY the schema below.
- Every field must exist. If unknown, write 'unknown'.

OUTPUT SCHEMA:

PAIN_LOCATION:
PAIN_TYPE:
TRIGGERS:
DURATION:
INTENSITY:
OTHER_NOTES:
"""


def generate_summary(history, prev_summary="", max_new_tokens=150):
    history_text = format_history_for_summary(history[-4:])

    # prima memorii: schema goală
    if not prev_summary:
        prev_summary = (
            "PAIN_LOCATION:\n"
            "PAIN_TYPE:\n"
            "TRIGGERS:\n"
            "DURATION:\n"
            "INTENSITY:\n"
            "OTHER_NOTES:"
        )

    prompt = f"""{SUMMARY_PROMPT}

PREVIOUS_MEMORY:
{prev_summary}

CONVERSATION_EXCERPT:
{history_text}

FINAL_MEMORY:
"""

    # Tokenizare
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generare – FOARTE IMPORTANT: fără sampling
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,        # determinist
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # Izolăm doar schema
    start = text.find("PAIN_LOCATION:")
    if start != -1:
        text = text[start:]

    cutoff = text.rfind("OTHER_NOTES:")
    if cutoff != -1:
        text = text[:cutoff + len("OTHER_NOTES:")]

    return text.strip()


In [54]:
def step(case: Case, state: State, user_msg: str) -> str:
    """Un pas de conversatie: doctorul pune o intrebare, pacientul (LLM) raspunde.
    Integreaza si un LLM mic pentru memorie (rezumatul conversatiei).
    """
    state.history.append({"role": "user", "content": user_msg})

    disease_key = case.diagnosis_truth
    system_prompt = build_system_prompt_for_case(disease_key, mapping)

    messages = [{"role": "system", "content": system_prompt}]

    messages += state.history

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            min_new_tokens=10,
            do_sample=True,
            top_p=0.6,
            temperature=0.15,
            repetition_penalty=1.05,
        )

    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    state.history.append({"role": "assistant", "content": reply})

    for s in case.symptoms_truth:
        if s.lower() in reply.lower():
            case.revealed_symptoms.add(s)

    state.memory_summary = generate_summary(
        state.history,
        prev_summary=state.memory_summary,
        max_new_tokens=150
    )

    print("\n===== SUMMARY UPDATE (DEBUG) =====")
    print(state.memory_summary)
    print("=================================\n")

    return reply


In [57]:
case: Case = None
state: State = None

def start_new_case():
    """Pornește un caz nou cu diagnostic aleator."""
    global case, state
    case = init_case(mapping)
    state = State()
    print("Caz nou inceput!")
    print("Scrie /help pentru lista de comenzi.")
    #print(f"(debug) Boala interna aleasa: {case.diagnosis_truth}")
    return case, state

HELP_TEXT = """Comenzi disponibile:
/help       - afiseaza aceasta lista
/truth      - afiseaza diagnosticul real + simptomele (debug)
/revealed   - arata simptomele reale deja dezvaluite
/new        - porneste un caz nou
/exit       - inchide sesiunea de chat
"""

def show_truth():
    print("Diagnostic ASCUNS:", case.diagnosis_truth)
    print("Simptome reale:", ", ".join(sorted(case.symptoms_truth)))

def show_revealed():
    print("REVEALED:", sorted(case.revealed_symptoms) or "(niciunul)")


def chat_loop():
    print("CONVERSATIE LIVE CU PACIENTUL")
    print("Scrie intrebarile tale (sau /help)")

    while True:
        try:
            user_msg = input("\nTu: ").strip()
        except EOFError:
            break
        if not user_msg:
            continue

        cmd = user_msg.lower()
        if cmd == "/help":
            print(HELP_TEXT); continue
        if cmd == "/exit":
            print("Inchis"); break
        if cmd == "/truth":
            show_truth(); continue
        if cmd == "/revealed":
            show_revealed(); continue
        if cmd == "/new":
            start_new_case(); continue
        if cmd == "/mem":
            print("\n===== SUMMARY MEMORY =====")
            print(state.memory_summary)
            print("==========================\n")
            continue


        try:
            raspuns = step(case, state, user_msg)
            print("Pacient:", raspuns)
        except Exception as e:
            print("Eroare in step():", e)


start_new_case()
chat_loop()


Caz nou inceput!
Scrie /help pentru lista de comenzi.
CONVERSATIE LIVE CU PACIENTUL
Scrie intrebarile tale (sau /help)

Tu: hello


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



===== SUMMARY UPDATE (DEBUG) =====
PAIN_LOCATION:
PAIN_TYPE:
TRIGGERS:
DURATION:
INTENSITY:
OTHER_NOTES:


PREVIOUS_MEMORY:
PAIN_LOCATION:
PAIN_TYPE:
TRIGGERS:
DURATION:
INTENSITY:
OTHER_NOTES:

CONVERSATION_EXCERPT:
[DOCTOR] hello
[PATIENT] I've been having some issues with my tooth. When I drink something cold, like ice water, it hurts.

FINAL_MEMORY:
PAIN_LOCATION:
PAIN_TYPE:
TRIGGERS:
DURATION:
INTENSITY:
OTHER_NOTES:

Pacient: I've been having some issues with my tooth. When I drink something cold, like ice water, it hurts.


KeyboardInterrupt: Interrupted by user